# Reinforcement learning: only rewards

```{admonition} Question
:class: tip
How does a robot get from 152 moves to 14?
```

**In this notebook:** 1 · a street grid  2 · one move  3 · learning by trying  4 · 300 tries  5 · the learned route

## 1 · A street grid

No table and no hexagons here. A courier robot, the **agent**, has to drive from a depot to a customer. The street grid is its **environment**. Nobody shows it a route.

In [ ]:
import numpy as np

import helpers as h

COLS, ROWS = 7, 5
DEPOT, CUSTOMER = (0, 0), (6, 4)
ROADWORKS = {(2, 0), (2, 1), (2, 2), (4, 4), (4, 3), (4, 2)}
MOVES = [(0, 1), (1, 0), (0, -1), (-1, 0)]          # north, east, south, west
streets = {"cols": COLS, "rows": ROWS, "depot": DEPOT, "customer": CUSTOMER, "roadworks": ROADWORKS}   # for the drawings
h.courier_streets(streets);

## 2 · One move and its points

The place where the robot stands is the **state**, a move is an **action**, and the points that follow are the **reward**: +20 for arriving, −10 for driving into roadworks, −1 for every other move. Short routes pay best.

In [ ]:
def drive(place, move):
    """What the street grid answers to one move: the new place, the points for it, and whether the robot has arrived."""
    nxt = (place[0] + MOVES[move][0], place[1] + MOVES[move][1])
    if nxt in ROADWORKS:
        return place, -10, False                        # drives into roadworks and stays where it was
    if not (0 <= nxt[0] < COLS and 0 <= nxt[1] < ROWS):
        return place, -1, False                         # the edge of the map: a wasted move
    return nxt, (20 if nxt == CUSTOMER else -1), nxt == CUSTOMER


print(drive(DEPOT, 1))       # east from the depot: a free street
print(drive((1, 0), 1))      # east again: roadworks

## 3 · Learning by trying

The robot keeps one number for every place and move: the points it expects from there on. After each move it corrects that number a little. This update rule is **Q-learning**. Early tries are mostly random moves, to find out what pays. Later tries use what was found.

In [ ]:
def learn(tries=300, seed=0, max_moves=200):
    """Trial and error. The robot keeps one number for every place and move: the points it expects from there on.
    After each move it corrects that number a little (the method is called Q-learning)."""
    rng = np.random.default_rng(seed)
    expect = np.zeros((COLS, ROWS, 4))
    totals, routes = [], []
    for t in range(tries):
        curious = max(0.02, 1 - t / (0.6 * tries))     # share of moves made at random: high at first, then falling
        place, total, route = DEPOT, 0, [DEPOT]
        for _ in range(max_moves):
            move = int(rng.integers(4)) if rng.random() < curious else int(np.argmax(expect[place]))
            nxt, points, arrived = drive(place, move)
            best_next = 0.0 if arrived else expect[nxt].max()
            expect[place][move] += 0.5 * (points + 0.95 * best_next - expect[place][move])     # the update rule
            place, total = nxt, total + points
            route.append(place)
            if arrived:
                break
        totals.append(total)
        routes.append(route)
    return np.array(totals), routes, expect

## 4 · 300 tries

One run of 300 tries. The lines show six of them, the pictures three.

In [ ]:
totals, routes, expect = learn()
for t in (1, 5, 20, 60, 150, 300):
    print(f"try {t:3d}: {len(routes[t - 1]) - 1:3d} moves, {totals[t - 1]:+5d} points")
h.courier_tries(streets, routes, totals, tries=(1, 60, 300));

<div class="frames" data-name="courier" data-labels="try 1|try 2|try 3|try 5|try 10|try 20|try 40|try 60|try 100|try 150|try 200|try 300"></div>

*The route of the robot on one try (left) and the points of all tries so far (right).*

**Result.** Try 1 takes 152 moves and collects −275 points. Try 300 takes 14 moves and collects +7. Nobody told the robot which move was wrong. It repeated the moves that were followed by more points.

## 5 · The learned route

The **policy** is what the robot has learned: in every place, take the move it expects most from. The last cell follows the policy without random moves, and repeats the 300 tries with 50 other seeds.

In [ ]:
def learned_route(expect, max_moves=60):
    """The route the robot drives when it always takes the move it expects most from."""
    place, route = DEPOT, [DEPOT]
    for _ in range(max_moves):
        place, _, arrived = drive(place, int(np.argmax(expect[place])))
        route.append(place)
        if arrived:
            break
    return route

In [ ]:
final = learned_route(expect)
print(f"after 300 tries, without random moves: {len(final) - 1} moves, {20 - (len(final) - 2):+d} points. The shortest safe route has 14 moves.")
repeats = np.array([len(learned_route(learn(seed=s)[2])) - 1 for s in range(50)])
print(f"Repeated 50 times with other random numbers: the learned route is the shortest possible in {int((repeats == 14).sum())} of 50.")

**Result.** 14 moves, the shortest safe route, in 50 of 50 repeats. No labelled examples were used, only points. The data did not exist beforehand: the robot's own moves produced them.

**Next:** [spatially explicit models](spatially_explicit).